# 1. Exploratory Data Analysis (EDA) - Klachten dataset

## 1.1 Packages importeren 

In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns 
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# Pandas display opties aanpassen
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

## 1.2 Dataset inladen

In [ ]:
df = pd.read_csv('klachten.csv')
y = df['Product'] # target 

In [ ]:
# Train-test split
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=y)

## 1.3 Basis analyse

In [ ]:
# Aantal kolommen en rijen in de train dataset
train_df.shape
print("De dataset bevat {} rijen en {} kolommen.".format(train_df.shape[0], train_df.shape[1]))

In [ ]:
# Eerste vijf rijen van de trainingsset bekijken
train_df.head().T

In [ ]:
# stats numerieke variabelen 
stats_num = train_df.describe(include='int64').T.reset_index().T
stats_num

In [ ]:
# stats categorische variabelen 
stats_cat = train_df.describe(include='object').T.reset_index()
stats_cat

In [ ]:
print("Aantal categorische kolommen:", train_df.select_dtypes(include='object').shape[1])
print("Aantal numerieke kolommen:", train_df.select_dtypes(include=['int64', 'float64']).shape[1])

In [ ]:
# Datatypes en null-waarden in de trainingsset bekijken
train_df.info()

#### Conclusie basis analyse
- Op 3 mei 2024 zijn de meeste klachten ingediend: 31 klachten 
<br>

- De productsector 'Incasso' heeft de meeste klachten: 2865
<br>
- Er zijn dubbele waarden in de kolom 'Omschrijving': er hoort eigenlijk geen omschrijving-tekst te zijn die vaker van 1 keer voorkomt. 
<br>
- "Closed with explanation" is het meest voorkomende antwoord van het bedrijf op een klacht: 8155 keer
<br>
Er zitten geen nullwaardes in de trainset. Alle variabelen behalve `ID` zijn van het object type (tekst)

## 1.4 Analyse: Product (target)

In [ ]:
# Verdeling van de Product-sectoren in de trainingsset
train_df['Product'].value_counts()

In [ ]:
counts_product = train_df['Product'].value_counts() # tellen hoeveel klachten per product sector

# plotten barchart 
ax = counts_product.plot(kind='bar', figsize=(10,5))
ax.set_xlabel("Product (sector)")
ax.set_ylabel("Aantal")
ax.set_title("Aantal klachten per product sector")

#### Conclusie Product analyse:
- De sector met de meeste klachten in de trainset is 'Incasso'

## 1.5 Analyse: Omschrijving

In [ ]:
# Statistische informatie over de lengte van de klachtenbeschrijvingen
exploratie_df = train_df.copy()
exploratie_df['len'] = exploratie_df['Omschrijving'].str.len()
exploratie_df['len'].describe().round(2)

In [ ]:
# Gemiddelde lengte van klachtenbeschrijvingen per product sector
exploratie_df.groupby('Product')['len'].mean().round(2)

In [ ]:
# Aantal duplicaten in omschrijving tellen
train_df['Omschrijving'].duplicated().sum()

In [ ]:
# Rijen met duplicaten selecteren
explore_duplicates = train_df.loc[
    train_df['Omschrijving'].duplicated(keep=False)
].copy()

# Sorteren zodat de duplicates bij elkaar staan 
explore_duplicates = explore_duplicates.sort_values('Omschrijving')

explore_duplicates

In [ ]:
# Rijen met duplicaten selecteren hele dataset (niet alleen train)
explore_duplicates = df.loc[
    df['Omschrijving'].duplicated(keep=False)
].copy()

explore_duplicates = explore_duplicates.sort_values('Omschrijving')

explore_duplicates

#### Conclusie Omschrijving Analyse:
- Er zijn in totaal 67 rubbele omschrijving rijen in de trainset. 
- Twee rijen hebben exact dezelfde omschrijving (zie afbeelding), maar zijn gecategoriseerd met verschillende productsectoren. Dat kan wijzen op verkeerd gelabelde data. TODO: Onderzoeken/navragen bij opdrachtgever

- Er zijn in totaal 122 dubbele omschrijving rijen in de hele dataset 

## 1.6 Analyse: Antwoord bedrijf

In [ ]:
train_df['Antwoord_bedrijf'].value_counts()

## 1.7 Analyse: Datum ontvangst 

In [ ]:
# datum string omzetten naar datetime
train_df['Datum_ontvangst'] = pd.to_datetime(train_df['Datum_ontvangst'])

In [ ]:
counts = train_df.groupby(train_df['Datum_ontvangst'].dt.to_period('M')).size() # groeperen op jaar-maand en aantallen per maand tellen 
counts.index = counts.index.to_timestamp() 

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(counts.index, counts.values)

# maand - jaar labels 
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.title("Klachten per maand")
plt.xlabel("Datum")
plt.ylabel("Aantal klachten")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# groeperen per maand en per product sector
counts_sector = (
    train_df.groupby([train_df['Datum_ontvangst'].dt.to_period('M'), 'Product']).size().unstack(fill_value=0))

# index terugzetten naar timestamp 
counts_sector.index = counts_sector.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12,5))

for sector in counts_sector.columns:
    ax.plot(counts_sector.index, counts_sector[sector], label=sector)

# Labels en titel
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y')) # maand - jaar labels 
plt.title("Klachten per maand per product sector")
plt.xlabel("Datum")
plt.ylabel("Aantal klachten")
plt.legend(title="Product sector")
plt.grid(True)
plt.tight_layout()
plt.show()

#### Conclusie Analyse: Datum ontvangst
Klachten per maand: 
- Het aantal klachten per maand schommelt, maar de grafiek laat wel een stijging in de loop van de tijd zien. 
- Er zijn vier momenten waarop het aantal klachten omhoog schiet: in juli 2023, oktober 2023, oktober 2024 en rond maart 2025.  

Klachten per maand per product sector:
- Incasso is duidelijk de grootste bron van klachten. Daarna volgen hypotheek en kredietregistratie. De andere productgroepen hebben het laagste aantal maandelijkse klachten. 

- Kredietregistratie en incasso laten een vergelijkbaar patroon zien: hun pieken en dalen liggen vaak dicht bij elkaar. Mogelijk hebben de klachten dezelfde of een soortgelijke oorzaak. 

## CONCLUSIE EDA VOOR DATA SELECTION
- We gebruiken de variabele `Omschrijving` als input en de `Product` als targetvariabele
- De overige variabelen voegen geen voorspellende waarde toe